In [1]:
import pandas as pd
from mopac_batch_runner import prepare_batch, run_mopac_batch, export_sdf

In [2]:
df = pd.read_excel('./datasets/pxr_dataset.xlsx')


df_100 = df.head(100)

In [3]:
df_100.head()

,SMILES,CASRN,EXTERNALID,N,NAME,ARTICLEID,PAGE,TABLE,Endocrine Disrupting-PXR,COMMENTS,INCHI_KEY
0,CCOP(=S)(OCC)OC1=CC=C(C=C1)[N+]([O-])=O,56-38-2,-,AUTO_1,-,A158580,-,-,positive,-,IAKIZGUCZWELFS-UHFFFAOYSA
1,COC(C)C(NC(=O)C(CC(C)C)C(CCC(F)(F)F)N(O)C=O)C(...,212609-69-3,-,AUTO_2,-,A158580,-,-,positive,-,QLGNVCMFFQTWAL-UHFFFAOYSA
2,CC(=O)C1CCC2C3CCC4CC(O)CCC4(C)C3C(=O)CC12C,23930-19-0,-,AUTO_3,-,A158580,-,-,positive,-,DUHUCHOQIDJXAT-UHFFFAOYSA
3,CC12CCC3C(CCC4=CC(=O)CCC34C)C1CCC2O,58-22-0,-,AUTO_4,-,A158580,-,-,positive,-,MUMGGOZAMZWBJJ-UHFFFAOYSA
4,CN1C(=O)N(C(=O)C=C1C(F)(F)F)C1=CC=C(Cl)C(=C1)C...,134605-64-4,-,AUTO_5,-,A158580,-,-,positive,-,JEDYYFXHPAIBGR-UHFFFAOYSA


In [17]:
# Step 1: SMILES → 3D structures → .mop files
prep = prepare_batch(
    df=df_100,
    smiles_col="smiles",
    id_col="compound_id",
    keywords="PM7 PRECISE GEO-OK",
    workdir="mopac_jobs",
    auto_charge=True,   # ← detects [NH4+], [O-], etc. automatically
    overwrite=True,
    max_workers=6,
)

In [18]:
# verify if anything failed
print(prep[prep['status'] != 'ready'])

Empty DataFrame
Columns: [compound_id, smiles, status, mop_file, error_message, duration_s]
Index: []


In [19]:
# Step 2: run MOPAC on all prepared .mop files
results = run_mopac_batch(
    workdir="mopac_jobs",
    overwrite=True,
    timeout=900,
    max_workers=6,
)

In [20]:
print(results[results['status'] != 'success'])

Empty DataFrame
Columns: [compound_id, smiles, status, mop_file, out_file, arc_file, error_message, return_code, duration_s]
Index: []


In [21]:
print(results[results['arc_file'].isna()])#.to_csv('failed_runs.csv')

Empty DataFrame
Columns: [compound_id, smiles, status, mop_file, out_file, arc_file, error_message, return_code, duration_s]
Index: []


In [26]:
#results

In [23]:
# Step 3: export optimised geometries to SDF
sdf = export_sdf(
    workdir="mopac_jobs",
    overwrite=True,
    max_workers=6,
)

In [24]:
print(sdf[sdf['status'] != 'success'])

Empty DataFrame
Columns: [compound_id, arc_file, sdf_file, status, error_message, duration_s]
Index: []


In [25]:
from pathlib import Path
import shutil

source_dir = Path(r"C:\Users\macie\Documents\GitHub\smi_to_mop\mopac_jobs")
target_dir = Path(r"C:\Users\macie\Documents\GitHub\smi_to_mop\sdf_all")

target_dir.mkdir(exist_ok=True)

for sdf_file in source_dir.rglob("*.sdf"):
    shutil.copy2(sdf_file, target_dir / sdf_file.name)